# 06 — Network accessibility


In [1]:
from pathlib import Path
import sys

import geopandas as gpd
import networkx as nx
import osmnx as ox
import pandas as pd

# ---------------------------------------------------------
# Resolve project root
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/subhankarbiswas/smart-city-knowledge-graph-v3


In [2]:
from src.config import settings
from src.network_accessibility import (
    add_walk_time,
    accessibility_from_origins,
)

print("Walking speed:", settings.walking_speed_kph, "km/h")
print("Maximum origins:", settings.accessibility_max_origins)

Walking speed: 5.0 km/h
Maximum origins: 750


In [3]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

GRAPH_PATH = RAW_DIR / "walk_network.graphml"
POIS_PATH = PROCESSED_DIR / "pois_clean.parquet"

ACCESSIBILITY_PATH = PROCESSED_DIR / "network_accessibility.csv"
SUMMARY_PATH = PROCESSED_DIR / "network_accessibility_summary.csv"

required_files = [
    GRAPH_PATH,
    POIS_PATH,
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "The following required files are missing:\n"
        + "\n".join(f"  - {path}" for path in missing_files)
    )

print("Required input files found:")
for path in required_files:
    print(f"  ✓ {path}")

Required input files found:
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/raw/walk_network.graphml
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/pois_clean.parquet


In [4]:
G = ox.load_graphml(GRAPH_PATH)

print("=" * 60)
print("WALKING NETWORK")
print("=" * 60)

print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"Directed: {G.is_directed()}")

WALKING NETWORK
Nodes: 4,122
Edges: 11,226
Directed: True


In [5]:
G = add_walk_time(
    G,
    settings.walking_speed_kph,
)

print("\nWalking-time weights added.")

# Inspect a few edges
edge_sample = list(G.edges(data=True))[:5]

for u, v, data in edge_sample:
    print(
        f"{u} -> {v}: "
        f"length={data.get('length')} m, "
        f"walk_time={data.get('walk_time')} s"
    )


Walking-time weights added.
26808663 -> 418721004: length=297.3953931677759 m, walk_time=None s
26808663 -> 418720998: length=171.99000372942658 m, walk_time=None s
26808663 -> 254403227: length=39.49177394642198 m, walk_time=None s
26808666 -> 2142727306: length=102.65719906243824 m, walk_time=None s
26808666 -> 9835146851: length=196.72890452887492 m, walk_time=None s


In [6]:
pois = gpd.read_parquet(POIS_PATH)

print("=" * 60)
print("POI DATASET")
print("=" * 60)

print(f"Total POIs: {len(pois):,}")
print(f"CRS: {pois.crs}")

if "service_type" not in pois.columns:
    raise KeyError(
        "Column 'service_type' is missing from pois_clean.parquet."
    )

print("\nService types:")
print(
    pois["service_type"]
    .fillna("unknown")
    .value_counts()
    .head(20)
)

POI DATASET
Total POIs: 224
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": 

In [7]:
pois = pois.copy()

pois["service_type"] = (
    pois["service_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Normalized service types:")
print(sorted(pois["service_type"].dropna().unique()))

Normalized service types:
['atm', 'bank', 'bus_stop', 'hospital', 'library', 'park', 'pharmacy', 'school', 'supermarket', 'train_station']


In [8]:
ORIGIN_SERVICES = [
    "school",
    "university",
    "supermarket",
]

TARGET_SERVICES = [
    "hospital",
    "pharmacy",
    "supermarket",
    "train_station",
    "library",
]

origin_candidates = pois[
    pois["service_type"].isin(ORIGIN_SERVICES)
].copy()

if origin_candidates.empty:
    raise ValueError(
        "No accessibility origins were found for: "
        f"{ORIGIN_SERVICES}"
    )

origins = origin_candidates.head(
    settings.accessibility_max_origins
).copy()

print("=" * 60)
print("ACCESSIBILITY ORIGINS")
print("=" * 60)

print(f"Origin candidates: {len(origin_candidates):,}")
print(f"Origins used:      {len(origins):,}")

print("\nOrigins by service:")
print(
    origins["service_type"]
    .value_counts()
)

ACCESSIBILITY ORIGINS
Origin candidates: 22
Origins used:      22

Origins by service:
service_type
school         12
supermarket    10
Name: count, dtype: int64


In [9]:
print("=" * 60)
print("TARGET SERVICES")
print("=" * 60)

target_counts = (
    pois[pois["service_type"].isin(TARGET_SERVICES)]
    ["service_type"]
    .value_counts()
    .reindex(TARGET_SERVICES, fill_value=0)
)

display(
    target_counts
    .rename("count")
    .to_frame()
)

TARGET SERVICES


,count
service_type,
hospital,1
pharmacy,5
supermarket,10
train_station,1
library,1


In [10]:
rows = []

print("=" * 60)
print("NETWORK ACCESSIBILITY ANALYSIS")
print("=" * 60)

for service in TARGET_SERVICES:

    target = pois[
        pois["service_type"] == service
    ].copy()

    print(f"\n{service}")
    print("-" * 40)
    print(f"Target POIs: {len(target):,}")

    if target.empty:
        print("No targets found — skipping.")
        continue

    try:
        result = accessibility_from_origins(
            G,
            origins,
            target,
        )

        if result is None or result.empty:
            print("No accessibility results returned.")
            continue

        result = result.copy()
        result["service"] = service

        rows.append(result)

        print(f"Results: {len(result):,}")

    except Exception as exc:
        print(f"Accessibility calculation failed: {exc}")

NETWORK ACCESSIBILITY ANALYSIS

hospital
----------------------------------------
Target POIs: 1
Results: 22

pharmacy
----------------------------------------
Target POIs: 5
Results: 22

supermarket
----------------------------------------
Target POIs: 10
Results: 22

train_station
----------------------------------------
Target POIs: 1
Results: 22

library
----------------------------------------
Target POIs: 1
Results: 22


In [11]:
if rows:
    network_accessibility = pd.concat(
        rows,
        ignore_index=True
    )
else:
    network_accessibility = pd.DataFrame()

print("=" * 60)
print("ACCESSIBILITY RESULTS")
print("=" * 60)

print(
    f"Total accessibility records: "
    f"{len(network_accessibility):,}"
)

if not network_accessibility.empty:
    display(network_accessibility.head(10))
else:
    print("No accessibility results were generated.")

ACCESSIBILITY RESULTS
Total accessibility records: 110


,origin_index,network_node,travel_time_s,travel_time_min,service
0,6,5144049364,1933.581874,32.226365,hospital
1,7,428473989,2343.797150,39.063286,hospital
2,8,428469705,2151.900526,35.865009,hospital
3,9,428473989,2343.797150,39.063286,hospital
4,10,254161663,715.177882,11.919631,hospital
5,11,28923280,596.674421,9.944574,hospital
6,12,1898284092,1063.308010,17.721800,hospital
7,13,2181649761,2176.655892,36.277598,hospital
8,14,1248399607,1668.147245,27.802454,hospital
9,15,2846150385,1204.700073,20.078335,hospital


In [12]:
if not network_accessibility.empty:
    
    print("Result columns:")
    for column in network_accessibility.columns:
        print(f"  - {column}")

    print("\nData types:")
    display(network_accessibility.dtypes.to_frame("dtype"))

Result columns:
  - origin_index
  - network_node
  - travel_time_s
  - travel_time_min
  - service

Data types:


,dtype
origin_index,int64
network_node,int64
travel_time_s,float64
travel_time_min,float64
service,str


In [13]:
if not network_accessibility.empty:
    
    summary = (
        network_accessibility
        .groupby("service")
        .size()
        .reset_index(name="records")
        .sort_values("records", ascending=False)
    )

    print("=" * 60)
    print("ACCESSIBILITY SUMMARY")
    print("=" * 60)

    display(summary)

ACCESSIBILITY SUMMARY


,service,records
0,hospital,22
1,library,22
2,pharmacy,22
3,supermarket,22
4,train_station,22


In [14]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

network_accessibility.to_csv(
    ACCESSIBILITY_PATH,
    index=False
)

print(f"Saved accessibility results to:")
print(ACCESSIBILITY_PATH)

Saved accessibility results to:
/Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/network_accessibility.csv


In [15]:
if not network_accessibility.empty:
    
    summary = (
        network_accessibility
        .groupby("service")
        .size()
        .reset_index(name="records")
        .sort_values("records", ascending=False)
    )

    summary.to_csv(
        SUMMARY_PATH,
        index=False
    )

    print(f"Saved summary to:")
    print(SUMMARY_PATH)

Saved summary to:
/Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/network_accessibility_summary.csv


In [16]:
print("=" * 60)
print("NOTEBOOK 06 VALIDATION")
print("=" * 60)

print(f"Walking network nodes:       {G.number_of_nodes():,}")
print(f"Walking network edges:       {G.number_of_edges():,}")
print(f"Total POIs:                  {len(pois):,}")
print(f"Accessibility origins used:  {len(origins):,}")
print(f"Accessibility records:       {len(network_accessibility):,}")

print("\nTarget services processed:")

for service in TARGET_SERVICES:
    count = (
        network_accessibility["service"].eq(service).sum()
        if not network_accessibility.empty
        else 0
    )
    print(f"  {service:<15} {count:,}")

print("\nOutput files:")

for path in [
    ACCESSIBILITY_PATH,
    SUMMARY_PATH,
]:
    if path.exists():
        print(f"  ✓ {path}")
    else:
        print(f"  ✗ {path}")

NOTEBOOK 06 VALIDATION
Walking network nodes:       4,122
Walking network edges:       11,226
Total POIs:                  224
Accessibility origins used:  22
Accessibility records:       110

Target services processed:
  hospital        22
  pharmacy        22
  supermarket     22
  train_station   22
  library         22

Output files:
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/network_accessibility.csv
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/network_accessibility_summary.csv
